# MOdel Training

## 1 Import Data and Required Packeges

In [78]:
# Basic Import
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
# Modelling
from sklearn.metrics import mean_squared_error,r2_score
from sklearn.linear_model import LinearRegression,Ridge,Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor,AdaBoostRegressor
from sklearn.metrics import mean_absolute_error,root_mean_squared_error
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
# Warnings
import warnings
warnings.filterwarnings('ignore')


## Import the CSV Data as Pandas Dataframe

In [79]:
df = pd.read_csv('data/stud.csv')

In [80]:
df.head()

,gender,race_ethnicity,parental_level_of_education,lunch,test_preparation_course,math_score,reading_score,writing_score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


In [81]:
# Prepare X and Y Feature
y = df['math_score']
x = df.drop(['math_score'],axis=1)

In [82]:
x.head()

,gender,race_ethnicity,parental_level_of_education,lunch,test_preparation_course,reading_score,writing_score
0,female,group B,bachelor's degree,standard,none,72,74
1,female,group C,some college,standard,completed,90,88
2,female,group B,master's degree,standard,none,95,93
3,male,group A,associate's degree,free/reduced,none,57,44
4,male,group C,some college,standard,none,78,75


In [83]:
df.nunique()

gender                          2
race_ethnicity                  5
parental_level_of_education     6
lunch                           2
test_preparation_course         2
math_score                     81
reading_score                  72
writing_score                  77
dtype: int64

In [84]:
# Create column Tranmsformer with # type of Transformer
num_features = x.select_dtypes(include=int).columns
cat_features = x.select_dtypes(include=str).columns

from sklearn import preprocessing
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.compose import ColumnTransformer

Num_Transformer = StandardScaler()
Cat_Transformer = OneHotEncoder()

preprocessor = ColumnTransformer(
    [
        ('OneHotEncoder',Cat_Transformer,cat_features),
        ('StandardScaler',Num_Transformer,num_features)
    ]
)


In [85]:
# Train Test Split
from numpy.random import RandomState
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(x,y,random_state=42,test_size=0.2)

x_train = preprocessor.fit_transform(x_train)
x_test = preprocessor.transform(x_test)

## Create an Evaluate Function to give all metrics after model training


In [86]:
def evaluate_model(true,predict):
    mae = mean_absolute_error(true,predict)
    mse = mean_squared_error(true,predict)
    rmse = root_mean_squared_error(true,predict)
    r2_square = r2_score(true,predict)
    return mae,rmse,r2_square

In [87]:
models = {
    'LinearRegression': LinearRegression(),
    'Lasso' : Lasso(),
    'Ridge' : Ridge(),
    'KNeighborsRegressor' : KNeighborsRegressor(),
    'DecisionTreeRegressor' : DecisionTreeRegressor(),
    'RandomForestRegressor' : RandomForestRegressor(),
    'XGBRegressor' : XGBRegressor(),
    'CatBoostRegressor' : CatBoostRegressor(verbose=False),
    'AdaBoostRegressor' : AdaBoostRegressor()
}
model_list = []
r2_list = []

for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(x_train,y_train)

    # Make Prediction
    y_train_pred = model.predict(x_train)
    y_test_pred = model.predict(x_test)

    # Evaluate Train and Test Dataset
    model_train_mae,model_train_rmse,model_train_r2 = evaluate_model(y_train,y_train_pred)
    model_test_mae,model_test_rmse,model_test_r2 = evaluate_model(y_test,y_test_pred)

    print(list(models.keys())[i])
    model_list.append(list(models.keys())[i])

    print('Model performance for Training set')
    print('- Root Mean Squared Error : {:.4f}'.format(model_train_rmse))
    print('-Mean Absolute Error : {:.4f}'.format(model_train_rmse))
    print('- R2 score : {:.4f}'.format(model_train_r2))

    print('-'*35)

    print('Model performance for Testing set')
    print('- Root Mean Squared Error : {:.4f}'.format(model_test_rmse))
    print('-Mean Absolute Error : {:.4f}'.format(model_test_rmse))
    print('- R2 score : {:.4f}'.format(model_test_r2))
    r2_list.append(model_test_r2)

    print('='*35)
    print('\n')


LinearRegression
Model performance for Training set
- Root Mean Squared Error : 5.3231
-Mean Absolute Error : 5.3231
- R2 score : 0.8743
-----------------------------------
Model performance for Testing set
- Root Mean Squared Error : 5.3940
-Mean Absolute Error : 5.3940
- R2 score : 0.8804


Lasso
Model performance for Training set
- Root Mean Squared Error : 6.5925
-Mean Absolute Error : 6.5925
- R2 score : 0.8072
-----------------------------------
Model performance for Testing set
- Root Mean Squared Error : 6.5173
-Mean Absolute Error : 6.5173
- R2 score : 0.8254


Ridge
Model performance for Training set
- Root Mean Squared Error : 5.3233
-Mean Absolute Error : 5.3233
- R2 score : 0.8743
-----------------------------------
Model performance for Testing set
- Root Mean Squared Error : 5.3904
-Mean Absolute Error : 5.3904
- R2 score : 0.8806


KNeighborsRegressor
Model performance for Training set
- Root Mean Squared Error : 5.6974
-Mean Absolute Error : 5.6974
- R2 score : 0.8560


In [88]:
pd.DataFrame(list(zip(model_list,r2_list)),columns=['ModelName','R2_Score']).sort_values(by='R2_Score',ascending=False)

,ModelName,R2_Score
2,Ridge,0.880592
0,LinearRegression,0.880433
7,CatBoostRegressor,0.851831
5,RandomForestRegressor,0.849951
8,AdaBoostRegressor,0.844924
6,XGBRegressor,0.827797
1,Lasso,0.825446
3,KNeighborsRegressor,0.785944
4,DecisionTreeRegressor,0.718191


In [89]:
pred_df = pd.DataFrame({'Actual value' : y_test,'Predicted Value' : y_test_pred,'Diffrence' : y_test-y_test_pred})

In [90]:
pred_df

,Actual value,Predicted Value,Diffrence
521,91,76.807882,14.192118
737,53,55.992167,-2.992167
740,80,78.005848,1.994152
660,74,77.954887,-3.954887
411,84,83.076923,0.923077
...,...,...,...
408,52,48.940141,3.059859
332,62,57.768657,4.231343
208,74,66.823529,7.176471
613,65,68.148148,-3.148148
